**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/Users/sumahegde.work@outlook.com/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "databricksmaster", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/{data_source}/"

print("Base Path:", base_path)


# define tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

Base Path: abfss://source@myownstoragesm.dfs.core.windows.net/processed/child/orders/


## Bronze

In [0]:
df = (
    spark.read
        .options(header=True, inferSchema=True)
        .format("csv")
        .option("recursiveFileLookup", "true")
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .withColumn("file_name", F.expr("_metadata.file_path"))
)

print("Total Rows:", df.count())

df.show(5)

Total Rows: 51810
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FOCT62720602|Tuesday, Septembe...|     ABC987|  25891301|     71.0|2026-05-07 10:11:...|abfss://source@my...|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891502|    125.0|2026-05-07 10:11:...|abfss://source@my...|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891403|    462.0|2026-05-07 10:11:...|abfss://source@my...|
|FOCT62720602|Tuesday, Septembe...|    INVALID|  25891601|    133.0|2026-05-07 10:11:...|abfss://source@my...|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891602|     79.0|2026-05-07 10:11:...|abfss://source@my...|
+------------+--------------------+-----------+----------+---------+--------------------+-----

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

## Silver

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show(2)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-05-07 10:13:...|abfss://source@my...|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-05-07 10:13:...|abfss://source@my...|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
only showing top 2 rows



**Transformations**

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.to_date(F.col("order_placement_date"), "yyyy/MM/dd"),
        F.to_date(F.col("order_placement_date"), "dd-MM-yyyy"),
        F.to_date(F.col("order_placement_date"), "dd/MM/yyyy"),
        F.to_date(F.col("order_placement_date"), "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 6. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-11-30|
+----------+----------+



**Join with products**

In [0]:
df_products = spark.table("databricksmaster.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

df_joined.show(5)

+-------------+--------------------+-----------+----------+---------+--------------------+--------------------+--------------------+
|     order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|        product_code|
+-------------+--------------------+-----------+----------+---------+--------------------+--------------------+--------------------+
|FAUG410101302|          2025-08-08|     789101|  25891103|    493.0|2026-05-07 10:13:...|abfss://source@my...|102628255d24304d6...|
|FAUG410101302|          2025-08-08|     789101|  25891203|    374.0|2026-05-07 10:13:...|abfss://source@my...|889c67757ece9c973...|
|FAUG410101302|          2025-08-08|     789101|  25891302|     44.0|2026-05-07 10:13:...|abfss://source@my...|d9ebd1ca64d23951a...|
|FAUG410101402|          2025-08-07|     789101|  25891101|    311.0|2026-05-07 10:13:...|abfss://source@my...|e91ba9d665f90254d...|
|FAUG410101402|          2025-08-07|     789101|  25891201|    442.0|

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Gold

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

+-------------+----------+-------------+--------------------+----------+-------------+
|     order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+-------------+----------+-------------+--------------------+----------+-------------+
|FAUG410101302|2025-08-08|       789101|102628255d24304d6...|  25891103|        493.0|
|FAUG410101302|2025-08-08|       789101|889c67757ece9c973...|  25891203|        374.0|
+-------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows



In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

creating New Table


## Merging with Parent company

- Note: We want data for monthly level but child data is on daily level

**Full Load**

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-08-08|102628255d24304d6...|       789101|        493.0|
|2025-08-08|889c67757ece9c973...|       789101|        374.0|
|2025-08-08|d9ebd1ca64d23951a...|       789101|         44.0|
|2025-08-07|e91ba9d665f90254d...|       789101|        311.0|
|2025-08-07|2e387cef1424d6e7b...|       789101|        442.0|
|2025-08-07|fe5a8036be4b9a787...|       789101|        239.0|
|2025-08-09|c68834ceaff15846b...|       789101|         23.0|
|2025-08-09|ee1f7df9cf660ef02...|       789101|        123.0|
|2025-08-08|451f7167b28a25bde...|       999999|        197.0|
|2025-08-07|e91ba9d665f90254d...|       789102|        333.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows



In [0]:
df_child.count()

40811

In [0]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-08-01|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|789303       |3358.0       |
|2025-08-01|3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345|789402       |564.0        |
|2025-08-01|3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345|789403       |868.0        |
|2025-07-01|716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742|789603       |1855.0       |
|2025-07-01|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|789903       |3917.0       |
+----------+----------------------------------------------------------------+-------------+-------------+
only showing top 5 rows



In [0]:
df_monthly.count()

3060

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]